# 第1章　张量（Tensor）基础

张量是 PyTorch 的主角数据结构。简单说就是 **「NumPy 数组 ＋ 能在 GPU 上运行 ＋ 能自动微分」**。
它统一表示标量(0维)・向量(1维)・矩阵(2维)以及更高维。

本章目标：能够**创建张量、查看形状、做运算、改变形状、送到 GPU**。

> **本笔记使用方法**
> - 从上到下依次运行单元格（Colab/Jupyter 都是 `Shift + Enter`）。
> - 代码**稍作修改、弄坏再修好**最能进步。每章末尾有练习。
> - 多数章节不需要 GPU。较重的章节（CNN）会说明用法。

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA (GPU) available:", torch.cuda.is_available())

## 1-1. 创建张量

看几种常见的创建方式。`torch.tensor` 从已有数值创建，`zeros/ones/randn/arange` 按规则生成。

In [ ]:
import torch

a = torch.tensor([1.0, 2.0, 3.0])        # 从列表创建
b = torch.zeros(2, 3)                     # 2x3 全零矩阵
c = torch.ones(2, 3)                      # 2x3 全一矩阵
d = torch.randn(2, 3)                     # 标准正态分布随机数 2x3
e = torch.arange(0, 10, 2)                # 0,2,4,6,8

print("a =", a)
print("b =\n", b)
print("d =\n", d)
print("e =", e)

## 1-2. 张量的属性：`shape` / `dtype` / `device`

- `shape`（形状）… 各维度的大小。**最重要**。九成报错都来自这里不一致。
- `dtype`（类型）… `float32` 是标准。整数是 `int64`(long)。
- `device`（位置）… `cpu` 或 `cuda`（GPU）。

In [ ]:
x = torch.randn(3, 4)
print("shape :", x.shape)     # torch.Size([3, 4])
print("ndim  :", x.ndim)      # 维度数 = 2
print("dtype :", x.dtype)     # torch.float32
print("device:", x.device)    # cpu

# 转换类型
xi = x.to(torch.int64)
print("int  :", xi.dtype)

## 1-3. 运算：逐元素・矩阵乘法・广播

- `+ - * /` 是**逐元素**（相同位置之间）。
- 矩阵乘法用 `@`（或 `torch.matmul`）。**和逐元素乘法 `*` 是两回事**。
- 形状不同也能自动对齐的机制叫**广播（broadcast）**。

In [ ]:
m = torch.tensor([[1., 2.],
                  [3., 4.]])
n = torch.tensor([[10., 20.],
                  [30., 40.]])

print("逐元素乘 m*n =\n", m * n)        # 按位置相乘
print("矩阵乘   m@n =\n", m @ n)         # 行×列

# 广播：行向量自动加到每一行
row = torch.tensor([100., 200.])
print("广播 m+row =\n", m + row)

### 常用聚合
`sum / mean / max` 等的关键是 `dim`（沿哪个轴汇总）。`dim=0` 是列方向（压掉行），`dim=1` 是行方向（压掉列）。

In [ ]:
t = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])
print("总和      :", t.sum())
print("按列(dim=0):", t.sum(dim=0))   # [5, 7, 9]
print("按行(dim=1):", t.sum(dim=1))   # [6, 15]
print("按行平均  :", t.mean(dim=1))

## 1-4. 改变形状：`reshape` / `view` / `squeeze` / `unsqueeze`

- `reshape(...)`（或 `view`）… 保持元素数不变改变形状。`-1` 表示"剩下的自动算出"。
- `unsqueeze(d)` … **增加**一个大小为1的维度（常用于添加 batch 维）。
- `squeeze()` … **删除**大小为1的维度。

In [ ]:
x = torch.arange(12)          # 1维 [0..11]
print("原始:", x.shape)
y = x.reshape(3, 4)           # 3x4
print("reshape:", y.shape)
z = x.reshape(2, -1)          # 2x6 （-1 自动算出6）
print("auto -1:", z.shape)

img = torch.randn(28, 28)     # 一张图（28x28）
batch = img.unsqueeze(0)      # 在最前面加维度 -> (1, 28, 28)
print("unsqueeze:", batch.shape)
print("squeeze  :", batch.squeeze().shape)

## 1-5. 索引与切片（和 NumPy 一样的感觉）

In [ ]:
x = torch.arange(12).reshape(3, 4)
print(x)
print("第1行     :", x[0])        # 第一行
print("第2列     :", x[:, 1])     # 所有行的第2列
print("子块    :\n", x[0:2, 1:3]) # 0..1行, 1..2列
print("条件筛选 :", x[x > 5])     # 只取大于5的元素

## 1-6. 与 NumPy 互通
PyTorch 和 NumPy 可以互相转换（在 CPU 上是共享内存的，改一个另一个也会变，注意）。

In [ ]:
import numpy as np
np_arr = np.array([1., 2., 3.])
t = torch.from_numpy(np_arr)     # numpy -> tensor
back = t.numpy()                 # tensor -> numpy
print(type(t), t)
print(type(back), back)

## 1-7. 送到 GPU（固定写法）
"有 GPU 就用 GPU，没有就用 CPU" 的自动选择写法，记住它代码就能到处跑。
**模型和数据要放在同一个 device** 是铁律（不一致会报错）。

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("使用设备:", device)

x = torch.randn(2, 3).to(device)   # 把数据送到 device
print(x.device)

## 练习 1
1. 用 `torch.arange(24)` 造一个 `(2, 3, 4)` 的张量，确认 `shape`。
2. 造一个 `(3,3)` 随机矩阵，观察**矩阵乘** `A @ A` 与**逐元素乘** `A * A` 的区别。
3. `(5, 1)` 的张量加 `(1, 4)` 的张量会得到什么形状？（广播）

In [ ]:
# 在这里写你自己的代码并运行
